# Experiment 5— AutoGluon on the preprocessed feature set

This experiment uses `exploration/metul/preprocessing.py` for all feature engineering. Categorical columns remain native categorical values; there is no manual label encoding. The validation split is chronological so later rents are not used to train or tune the validation model.

In [2]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'), Path('/content/drive/MyDrive/KDDM-project')]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))
from preprocessing import load_datasets, preprocess_train_test

try:
    from autogluon.tabular import TabularPredictor
except ImportError as exc:
    raise ImportError('AutoGluon is required. Install it in the active environment with: pip install autogluon.tabular') from exc

RANDOM_STATE = 42
TARGET = 'MONTHLY_RENT'
TIME_LIMIT_SECONDS = int(os.environ.get('AUTOGLUON_TIME_LIMIT', '600'))
data_dir, datasets, train_df, test_df = preprocess_train_test(project_root / 'data')
print(f'Train shape: {train_df.shape}; test shape: {test_df.shape}')
print(f'Preprocessed columns: {len(train_df.columns)}')

Train shape: (150000, 34); test shape: (50000, 33)
Preprocessed columns: 34


## Prepare native feature types

All columns produced by the preprocessing module are retained except the target. The approval-month string is converted to a datetime because AutoGluon can derive calendar features from it. Other text columns are pandas categoricals, allowing AutoGluon to handle them natively.

In [3]:
def prepare_features(frame):
    result = frame.drop(columns=[TARGET], errors='ignore').copy()
    if 'RENT_APPROVAL_DATE' in result:
        result['RENT_APPROVAL_DATE'] = pd.to_datetime(result['RENT_APPROVAL_DATE'] + '-01')
    return result

X_full = prepare_features(train_df)
X_test = prepare_features(test_df)
y_full = train_df[TARGET].astype(float)

# Establish identical categorical vocabularies without using any target values.
for column in X_full.select_dtypes(include=['object', 'string', 'category']).columns:
    categories = pd.concat([X_full[column], X_test[column]], ignore_index=True).fillna('Unknown').astype(str).unique()
    X_full[column] = pd.Categorical(X_full[column].fillna('Unknown').astype(str), categories=categories)
    X_test[column] = pd.Categorical(X_test[column].fillna('Unknown').astype(str), categories=categories)

# Chronological holdout: the last 20% of observed training months is never
# passed to the validation model during fitting or model selection.
periods = pd.to_datetime(train_df['RENT_APPROVAL_DATE'] + '-01').sort_values().unique()
cutoff = periods[int(len(periods) * 0.80)]
is_train = pd.to_datetime(train_df['RENT_APPROVAL_DATE'] + '-01') < cutoff
X_train, X_valid = X_full.loc[is_train].copy(), X_full.loc[~is_train].copy()
y_train, y_valid = y_full.loc[is_train], y_full.loc[~is_train]
print(f'Chronological cutoff: {pd.Timestamp(cutoff).date()}')
print(f'Model rows: {len(X_train):,}; validation rows: {len(X_valid):,}')
print(f'Categorical columns: {X_full.select_dtypes(include=["category"]).columns.tolist()}')

Chronological cutoff: 2024-05-01
Model rows: 117,988; validation rows: 32,012
Categorical columns: ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FURNISHED', 'REGION']


## Validation fit and evaluation

`tuning_data` is explicitly supplied as the later-period holdout. AutoGluon is allowed to select and ensemble tabular models using only the earlier training rows and this declared validation set.

In [4]:
validation_predictor = TabularPredictor(
    label=TARGET,
    problem_type='regression',
    eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_model_validation'),
)
validation_train = X_train.copy()
validation_train[TARGET] = y_train.to_numpy()
validation_data = X_valid.copy()
validation_data[TARGET] = y_valid.to_numpy()
validation_predictor.fit(
    train_data=validation_train,
    tuning_data=validation_data,
    presets='best_quality',
    time_limit=TIME_LIMIT_SECONDS,
    verbosity=2,
)
valid_predictions = validation_predictor.predict(X_valid)
validation_metrics = pd.DataFrame([{
    'RMSE': mean_squared_error(y_valid, valid_predictions) ** 0.5,
    'MAE': mean_absolute_error(y_valid, valid_predictions),
    'R2': r2_score(y_valid, valid_predictions),
}])
display(validation_metrics)
display(validation_predictor.leaderboard(validation_data, silent=True))

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.13.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.32 GB / 16.00 GB (33.3%)
Disk Space Avail:   202.54 GB / 460.38 GB (44.0%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whethe

AssertionError: Learner is already fit.

## Final fit and test predictions

After validation is recorded, a separate AutoGluon predictor is fit on every labeled training row. The test set is never used during fitting; it is used only for prediction.

In [5]:
full_training_data = X_full.copy()
full_training_data[TARGET] = y_full.to_numpy()
final_predictor = TabularPredictor(
    label=TARGET,
    problem_type='regression',
    eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_model_final'),
)
final_predictor.fit(
    train_data=full_training_data,
    presets='best_quality',
    time_limit=TIME_LIMIT_SECONDS,
    verbosity=2,
)
test_predictions = final_predictor.predict(X_test).astype(float).clip(lower=0)
submission = test_df.copy()
submission['PREDICTED_MONTHLY_RENT'] = test_predictions.to_numpy()
output_path = experiment_dir / 'exp_5_autogluon_predictions.csv'
submission.to_csv(output_path, index=False)
print(f'Saved {len(submission):,} predictions to {output_path}')
display(submission[['RENT_APPROVAL_DATE', 'TOWN', 'FLAT_TYPE', 'PREDICTED_MONTHLY_RENT']].head())

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.13.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.37 GB / 16.00 GB (33.6%)
Disk Space Avail:   202.54 GB / 460.38 GB (44.0%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whethe

: 

In [4]:
from autogluon.tabular import TabularPredictor
import pandas as pd

predictor = TabularPredictor.load(
    "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_model_final/ds_sub_fit/sub_fit_ho"
)

predictions = predictor.predict(X_test).astype(float)

submission = pd.DataFrame({
    "Id": test_df.index,
    "Predicted": predictions.to_numpy(),
})

output_path = experiment_dir / "exp_5_autogluon_predictions.csv"
submission.to_csv(output_path, index=False)

print(f"Saved predictions to: {output_path}")
display(submission.head())

Saved predictions to: /Users/metul/Desktop/KDDM-project/exploration/metul/exp_5_autogluon_predictions.csv


,Id,Predicted
0,0,2722.022461
1,1,3691.018311
2,2,3053.918213
3,3,3122.218018
4,4,3290.280518
